# TP 8 — Kafka : partitions, offsets et groupes### Module 5 — Traitement temps réel**Durée :** 2 heures · **Noté sur 20**---## Ce que vous devez savoir faire à la fin1. Créer un topic et choisir son nombre de partitions **en le justifiant**.2. Produire avec et sans clé, et **démontrer** l'effet sur l'ordre.3. Lire les offsets d'un groupe et mesurer son retard (*lag*).4. Montrer que deux groupes lisent le même flux indépendamment.5. Provoquer un rééquilibrage et en observer le coût.6. Provoquer une perte et un doublon, en déplaçant la validation d'offset.## Barème| Exercice | Sujet | Points ||---|---|---|| 1 | Créer un topic | 2 || 2 | Produire : l'effet de la clé | 5 || 3 | Consommer : offsets et retard | 4 || 4 | Deux groupes, un topic | 3 || 5 | Rééquilibrage | 3 || 6 | Perte ou doublon | 3 |## Avant de commencerLe cluster doit tourner. Vérifiez que le broker répond :

In [ ]:
# 0 — Vérificationimport subprocess, json, timefrom datetime import datetimeKAFKA = "kafka:9092"BIN   = "/opt/kafka/bin"          # binaires Kafka dans le conteneur kafkadef kafka(cmd, conteneur="kafka"):    """Exécute une commande Kafka dans le conteneur kafka."""    r = subprocess.run(["bash", "-lc", cmd], capture_output=True, text=True)    return (r.stdout + r.stderr).strip()# Le client Python suffit pour l'essentiel du TPfrom kafka import KafkaProducer, KafkaConsumer, TopicPartitionfrom kafka.admin import KafkaAdminClient, NewTopicadmin = KafkaAdminClient(bootstrap_servers=KAFKA)print("Topics existants :", admin.list_topics())

> **Si l'import `kafka` échoue**, le paquet s'appelle `kafka-python-ng` et il est installé> dans l'image. Vérifiez avec `!pip show kafka-python-ng`.

---# Exercice 1 — Créer un topic  *(2 points)*

In [ ]:
# 1.1 — Un topic à 4 partitionsTOPIC = "transactions"try:    admin.delete_topics([TOPIC]); time.sleep(3)except Exception:    passadmin.create_topics([NewTopic(name=TOPIC, num_partitions=4, replication_factor=1)])time.sleep(2)print("Topics :", admin.list_topics())desc = admin.describe_topics([TOPIC])[0]for p in desc["partitions"]:    print(f"  partition {p['partition']} — leader broker {p['leader']}")

### Q1 *(2 pts)* —- **a.** Le facteur de réplication est de 1. Pourquoi, sur ce cluster ? Quelle valeur  utiliseriez-vous en production, et pourquoi ?- **b.** Le topic a 4 partitions. Quel est le nombre maximal de consommateurs utiles dans un  même groupe ? Que font les consommateurs au-delà ?

**Votre réponse :***(rédigez ici)*

---# Exercice 2 — Produire : l'effet de la clé  *(5 points)***Objectif.** Démontrer expérimentalement que l'ordre n'est garanti que par la clé.

## 2.1 — **PRÉDICTION** *(1 pt)*On va produire 2 000 messages concernant 5 comptes, **d'abord sans clé**, puis **avec`id_compte` comme clé**.Dans chaque cas : les messages d'un même compte se retrouveront-ils dans la même partition ?Leur ordre relatif sera-t-il préservé à la lecture ?

**Votre réponse :***(rédigez ici)*

In [ ]:
# 2.2 — Produire SANS cléproducteur = KafkaProducer(    bootstrap_servers=KAFKA,    value_serializer=lambda v: json.dumps(v).encode(),    acks="all")COMPTES = [f"C{i:04d}" for i in range(5)]for i in range(2000):    compte = COMPTES[i % 5]    producteur.send(TOPIC, value={        "seq": i, "id_compte": compte,        "montant": round(10 + (i % 97) * 1.7, 2),        "horodatage": datetime.utcnow().isoformat()})producteur.flush()print("2000 messages produits SANS cle")

In [ ]:
# 2.3 — Où sont-ils allés ?def repartition(topic, depuis=0):    """Compte les messages par partition, et par compte dans chaque partition."""    c = KafkaConsumer(bootstrap_servers=KAFKA, auto_offset_reset="earliest",                      enable_auto_commit=False, consumer_timeout_ms=4000,                      value_deserializer=lambda v: json.loads(v.decode()))    parts = [TopicPartition(topic, p) for p in c.partitions_for_topic(topic)]    c.assign(parts)    for tp in parts:        c.seek(tp, depuis)    tableau = {}    for msg in c:        tableau.setdefault(msg.partition, {}).setdefault(            msg.value["id_compte"], []).append(msg.value["seq"])    c.close()    return tableaut = repartition(TOPIC)for p in sorted(t):    comptes = {k: len(v) for k, v in t[p].items()}    print(f"partition {p} : {sum(comptes.values()):>5} messages, comptes {comptes}")

In [ ]:
# 2.4 — Produire AVEC clé, dans un topic neufTOPIC_CLE = "transactions-cle"try:    admin.delete_topics([TOPIC_CLE]); time.sleep(3)except Exception:    passadmin.create_topics([NewTopic(name=TOPIC_CLE, num_partitions=4, replication_factor=1)])time.sleep(2)producteur2 = KafkaProducer(    bootstrap_servers=KAFKA,    key_serializer=lambda k: k.encode(),    value_serializer=lambda v: json.dumps(v).encode(),    acks="all")for i in range(2000):    compte = COMPTES[i % 5]    producteur2.send(TOPIC_CLE, key=compte, value={        "seq": i, "id_compte": compte,        "montant": round(10 + (i % 97) * 1.7, 2),        "horodatage": datetime.utcnow().isoformat()})producteur2.flush()t2 = repartition(TOPIC_CLE)for p in sorted(t2):    comptes = {k: len(v) for k, v in t2[p].items()}    print(f"partition {p} : {sum(comptes.values()):>5} messages, comptes {comptes}")

In [ ]:
# 2.5 — L'ordre est-il préservé ?def ordre_preserve(tableau):    """Pour chaque compte, vérifie que ses seq sont croissants dans chaque partition."""    resultat = {}    for p, comptes in tableau.items():        for compte, seqs in comptes.items():            ok = seqs == sorted(seqs)            resultat.setdefault(compte, []).append((p, ok, len(seqs)))    return resultatprint("=== SANS cle ===")for compte, infos in sorted(ordre_preserve(t).items()):    parts = [p for p, _, _ in infos]    print(f"  {compte} : present dans les partitions {parts}")print("\n=== AVEC cle ===")for compte, infos in sorted(ordre_preserve(t2).items()):    parts = [p for p, _, _ in infos]    tri = all(ok for _, ok, _ in infos)    print(f"  {compte} : partitions {parts}, ordre croissant : {tri}")

### Q2 *(4 pts)* —- **a.** Sans clé, dans combien de partitions chaque compte apparaît-il ? Avec clé ?- **b.** Avec clé, deux comptes partagent-ils une partition ? Est-ce un problème ?- **c.** Un consommateur lit les 4 partitions. Sans clé, peut-il reconstituer l'ordre des  opérations d'un compte ? Justifiez.- **d.** Une application calcule le solde d'un compte en appliquant les mouvements dans  l'ordre. Laquelle des deux configurations est utilisable ? Que se passerait-il avec l'autre ?

**Votre réponse :***(rédigez ici)*

---# Exercice 3 — Consommer : offsets et retard  *(4 points)*

In [ ]:
# 3.1 — Un consommateur qui traite lentementGROUPE = "groupe-solde"c = KafkaConsumer(TOPIC_CLE, bootstrap_servers=KAFKA, group_id=GROUPE,                  auto_offset_reset="earliest", enable_auto_commit=True,                  consumer_timeout_ms=6000,                  value_deserializer=lambda v: json.loads(v.decode()))lus = 0for msg in c:    lus += 1    if lus >= 500:        breakc.commit()print(f"{lus} messages lus et valides")positions = {tp.partition: c.position(tp) for tp in c.assignment()}print("positions :", positions)c.close()

In [ ]:
# 3.2 — Mesurer le retard du groupedef retard(topic, groupe):    c = KafkaConsumer(bootstrap_servers=KAFKA, group_id=groupe,                      enable_auto_commit=False)    parts = [TopicPartition(topic, p) for p in c.partitions_for_topic(topic)]    c.assign(parts)    fins = c.end_offsets(parts)    total_lag, detail = 0, {}    for tp in parts:        engage = c.committed(tp) or 0        lag = fins[tp] - engage        detail[tp.partition] = (engage, fins[tp], lag)        total_lag += lag    c.close()    return total_lag, detailtotal, detail = retard(TOPIC_CLE, GROUPE)print(f"{'partition':>10} {'engage':>8} {'fin':>8} {'retard':>8}")for p, (e, f, l) in sorted(detail.items()):    print(f"{p:>10} {e:>8} {f:>8} {l:>8}")print(f"\nretard total du groupe : {total}")

### Q3 *(4 pts)* —- **a.** Qu'est-ce que le *retard* (*lag*) d'un groupe, en une phrase ?- **b.** Quel retard avez-vous mesuré ? Est-il réparti uniformément entre partitions ?- **c.** En production, on surveille cette métrique en permanence. Que signifie un retard  **stable et élevé** ? Et un retard **qui croît linéairement** ?- **d.** Un collègue propose de « remettre le retard à zéro » en supprimant le groupe.  Quelle en serait la conséquence ?

**Votre réponse :***(rédigez ici)*

---# Exercice 4 — Deux groupes, un topic  *(3 points)***Objectif.** Vérifier le découplage — la propriété qui fait tout l'intérêt de Kafka.

In [ ]:
# 4.1 — Deux groupes lisent le même topicdef lire_tout(topic, groupe, limite=3000):    c = KafkaConsumer(topic, bootstrap_servers=KAFKA, group_id=groupe,                      auto_offset_reset="earliest", enable_auto_commit=True,                      consumer_timeout_ms=5000,                      value_deserializer=lambda v: json.loads(v.decode()))    n = 0    for _ in c:        n += 1        if n >= limite:            break    c.commit(); c.close()    return nn_fraude    = lire_tout(TOPIC_CLE, "groupe-fraude")n_entrepot  = lire_tout(TOPIC_CLE, "groupe-entrepot")print(f"groupe-fraude   : {n_fraude} messages")print(f"groupe-entrepot : {n_entrepot} messages")print(f"groupe-solde    : {500} messages (lus a l'exercice 3)")

In [ ]:
# 4.2 — Les retards des trois groupesfor g in ["groupe-solde", "groupe-fraude", "groupe-entrepot"]:    total, _ = retard(TOPIC_CLE, g)    print(f"{g:<18} retard = {total}")

### Q4 *(3 pts)* —- **a.** Chaque groupe a-t-il reçu **tous** les messages, ou se les sont-ils partagés ?- **b.** Les trois groupes ont des retards différents. Que cela démontre-t-il ?- **c.** Une équipe propose de mettre les trois applications — fraude, solde, entrepôt — dans  un **seul** groupe, « pour économiser des ressources ». Expliquez précisément ce qui se  passerait.

**Votre réponse :***(rédigez ici)*

---# Exercice 5 — Le rééquilibrage  *(3 points)*

In [ ]:
# 5.1 — Un premier consommateur prend toutes les partitionsc1 = KafkaConsumer(TOPIC_CLE, bootstrap_servers=KAFKA, group_id="groupe-rebalance",                   auto_offset_reset="earliest", enable_auto_commit=True,                   consumer_timeout_ms=3000,                   value_deserializer=lambda v: json.loads(v.decode()))for _ in c1:    breakprint("c1 partitions :", sorted(p.partition for p in c1.assignment()))

In [ ]:
# 5.2 — Un second consommateur rejoint le groupec2 = KafkaConsumer(TOPIC_CLE, bootstrap_servers=KAFKA, group_id="groupe-rebalance",                   auto_offset_reset="earliest", enable_auto_commit=True,                   consumer_timeout_ms=3000,                   value_deserializer=lambda v: json.loads(v.decode()))debut = time.time()for _ in c2:    breakduree = time.time() - debut# Forcer c1 à re-participer pour voir sa nouvelle attributionfor _ in c1:    breakprint(f"reequilibrage observe en ~{duree:.1f} s")print("c1 partitions :", sorted(p.partition for p in c1.assignment()))print("c2 partitions :", sorted(p.partition for p in c2.assignment()))

In [ ]:
# 5.3 — Un cinquième consommateur, sur 4 partitionsautres = []for i in range(3):    ci = KafkaConsumer(TOPIC_CLE, bootstrap_servers=KAFKA, group_id="groupe-rebalance",                       auto_offset_reset="earliest", enable_auto_commit=True,                       consumer_timeout_ms=2000)    for _ in ci:        break    autres.append(ci)for i, ci in enumerate([c1, c2] + autres, start=1):    print(f"consommateur {i} : {sorted(p.partition for p in ci.assignment())}")for ci in [c1, c2] + autres:    ci.close()

### Q5 *(3 pts)* —- **a.** Comment les 4 partitions étaient-elles réparties avec 1, puis 2 consommateurs ?- **b.** Avec 5 consommateurs pour 4 partitions, que reçoit le cinquième ?- **c.** Le rééquilibrage a un coût : la consommation s'interrompt. Quelle conséquence pour  une application temps réel dont les consommateurs redémarrent souvent ?

**Votre réponse :***(rédigez ici)*

---# Exercice 6 — Perte ou doublon  *(3 points)***Objectif.** Provoquer les deux, en déplaçant le moment de la validation.

In [ ]:
# 6.1 — Validation AVANT traitement : provoquer une PERTEGROUPE_P = "groupe-perte"c = KafkaConsumer(TOPIC_CLE, bootstrap_servers=KAFKA, group_id=GROUPE_P,                  auto_offset_reset="earliest", enable_auto_commit=False,                  consumer_timeout_ms=4000,                  value_deserializer=lambda v: json.loads(v.decode()))traites, n = [], 0try:    for msg in c:        c.commit()                       # 1. VALIDER d'abord        if n == 20:            raise RuntimeError("PLANTAGE simule avant traitement")        traites.append(msg.value["seq"])  # 2. traiter ensuite        n += 1except RuntimeError as e:    print(e)c.close()print(f"traites avant plantage : {len(traites)}")

In [ ]:
# 6.2 — Redémarrage : que reprend-on ?c = KafkaConsumer(TOPIC_CLE, bootstrap_servers=KAFKA, group_id=GROUPE_P,                  auto_offset_reset="earliest", enable_auto_commit=False,                  consumer_timeout_ms=4000,                  value_deserializer=lambda v: json.loads(v.decode()))apres = []for msg in c:    apres.append(msg.value["seq"])    if len(apres) >= 5:        breakc.commit(); c.close()print("derniers seq traites avant plantage :", traites[-3:] if traites else "aucun")print("premiers seq apres redemarrage      :", apres[:3])print("\n-> le message en cours au moment du plantage a-t-il ete traite ?")

In [ ]:
# 6.3 — Validation APRÈS traitement : provoquer un DOUBLONGROUPE_D = "groupe-doublon"c = KafkaConsumer(TOPIC_CLE, bootstrap_servers=KAFKA, group_id=GROUPE_D,                  auto_offset_reset="earliest", enable_auto_commit=False,                  consumer_timeout_ms=4000,                  value_deserializer=lambda v: json.loads(v.decode()))traites2, n = [], 0try:    for msg in c:        traites2.append(msg.value["seq"])   # 1. traiter d'abord        n += 1        if n == 20:            raise RuntimeError("PLANTAGE simule avant validation")        c.commit()                          # 2. valider ensuiteexcept RuntimeError as e:    print(e)c.close()c = KafkaConsumer(TOPIC_CLE, bootstrap_servers=KAFKA, group_id=GROUPE_D,                  auto_offset_reset="earliest", enable_auto_commit=False,                  consumer_timeout_ms=4000,                  value_deserializer=lambda v: json.loads(v.decode()))apres2 = []for msg in c:    apres2.append(msg.value["seq"])    if len(apres2) >= 5:        breakc.close()communs = set(traites2) & set(apres2)print("derniers seq traites avant plantage :", traites2[-3:])print("premiers seq apres redemarrage      :", apres2[:3])print("seq traites DEUX FOIS               :", sorted(communs))

### Q6 *(3 pts)* —- **a.** Dans le premier cas (6.1–6.2), un message a-t-il été perdu ? Lequel, et pourquoi ?- **b.** Dans le second (6.3), un message a-t-il été traité deux fois ? Lequel ?- **c.** Quelle sémantique correspond à chaque cas ? Laquelle choisiriez-vous pour un  traitement de mouvements bancaires, et **que devez-vous faire en plus** pour que ce choix  soit correct ?

**Votre réponse :***(rédigez ici)*

---# Synthèse| Question | Votre réponse ||---|---|| Que garantit la clé de partitionnement ? | || Quel est le parallélisme maximal d'un groupe ? | || Que signifie un retard qui croît linéairement ? | || Pourquoi un groupe par application ? | || *At least once* + quoi = effet *exactly once* ? | |

In [ ]:
# Nettoyagefor t in [TOPIC, TOPIC_CLE]:    try:        admin.delete_topics([t])    except Exception as e:        print(t, e)print("Topics supprimes.")

---## Avant de rendre- [ ] Les **deux prédictions** (2.1 et votre réponse Q1b) sont écrites avant exécution.- [ ] Les questions **Q1 à Q6** sont rédigées et justifiées.- [ ] Les répartitions par partition sont reportées dans vos réponses.- [ ] Le tableau de synthèse est complété.- [ ] Notebook exporté en HTML et déposé.**Bon TP.**